# Notebook 01 — Data Extraction
**Project:** Ola Bengaluru Rides — Cancellation Intelligence & Revenue Recovery  
**Dataset:** Bengaluru Ola Rides, January 2024  
**Author:** Joshit  
**Institute:** Newton School of Technology — DVA Capstone 2

This notebook is just for loading and understanding the raw data. No changes are made to the original file here — all transformations happen in the cleaning notebook.

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries ready.')

Libraries ready.


In [20]:
# update path if running locally: '../data/raw/Bengaluru_Ola.csv'
RAW_PATH = '/content/Bengaluru Ola (1).csv'

df = pd.read_csv(RAW_PATH)
print(f'loaded: {df.shape[0]} rows, {df.shape[1]} columns')

loaded: 49999 rows, 21 columns


In [21]:
df.head()

In [22]:
df.dtypes

In [23]:
df.info()

### Checking nulls
Wanted to understand where the missing values are and why — turns out most nulls make sense structurally (cancelled rides won't have a booking value, ratings etc.)

In [24]:
null_df = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
null_df[null_df['null_count'] > 0].sort_values('null_count', ascending=False)

In [25]:
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 5))
bars = plt.bar(null_counts.index, null_counts.values, color='#E05C5C', edgecolor='white')
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.title('Missing Values by Column', fontsize=13, fontweight='bold')
plt.ylabel('Null Count')
for bar, val in zip(bars, null_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{val:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/01_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

### Looking at the categorical columns
Checking unique values for booking status, vehicle type, payment method and cancellation reasons — want to see if there are any inconsistencies before cleaning.

In [26]:
cat_cols = ['Booking Status', 'Vehicle Type', 'Payment Method',
            'Reason for Cancelling by Customer', 'Reason for Cancelling by Driver']

for col in cat_cols:
    print(f'\n{col}')
    print(df[col].value_counts(dropna=False))

### Quick stats on numeric columns

In [27]:
num_cols = ['Avg VTAT', 'Avg CTAT', 'Booking Value', 'Ride Distance', 'Driver Ratings', 'Customer Rating']
df[num_cols].describe().round(2)

### Booking status breakdown
This is the core of our analysis — understanding how many rides actually completed vs got cancelled.

In [28]:
status_counts = df['Booking Status'].value_counts()

for status, count in status_counts.items():
    pct = count / len(df) * 100
    print(f'{status:<30} {count:>6,}  ({pct:.1f}%)')

print(f'\nTotal non-successful: {len(df) - status_counts.get("Success", 0):,}')

In [29]:
colors = ['#2ECC71', '#E74C3C', '#E67E22', '#F39C12']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(status_counts.index, status_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Ride Count by Booking Status', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Rides')
axes[0].tick_params(axis='x', rotation=15)
for i, (idx, val) in enumerate(status_counts.items()):
    axes[0].text(i, val + 200, f'{val:,}', ha='center', fontsize=10)

axes[1].pie(status_counts.values, labels=status_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 10})
axes[1].set_title('Booking Status Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/01_booking_status.png', dpi=150, bbox_inches='tight')
plt.show()

In [32]:
print(f'Fully duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate Booking IDs: {df["Booking ID"].duplicated().sum()}')
print('flagging duplicate booking IDs for cleaning notebook')

Fully duplicate rows: 0
Duplicate Booking IDs: 133
flagging duplicate booking IDs for cleaning notebook


### Observations before cleaning

A few things I noticed going through the data:

- 33% of rides never completed — that's a big number and the core of our problem statement
- Driver cancellations (19.2%) are way higher than customer cancellations (7.6%) — interesting
- Nulls in booking value, ratings, distance are all structurally expected — only successful rides have these filled in, so I won't impute them
- `Cancelled by Customer` column has a double space in the name — needs fixing
- Date is a string in DD/MM/YYYY format — needs to be parsed properly
- 133 duplicate Booking IDs found — need to investigate in cleaning notebook
- Vehicle types look evenly distributed (~7000 each) which is good
- All locations follow Area-1 to Area-49 format — consistent

In [34]:
summary = {
    'Metric': ['Total Rows', 'Total Columns', 'Successful Rides', 'Driver Cancellations',
               'Customer Cancellations', 'Incomplete Rides', 'Total Nulls',
               'Duplicate Booking IDs', 'Avg Booking Value (INR)', 'Avg Ride Distance (km)'],
    'Value': [
        f"{len(df):,}",
        str(df.shape[1]),
        f"{(df['Booking Status']=='Success').sum():,}",
        f"{(df['Booking Status']=='Cancelled by Driver').sum():,}",
        f"{(df['Booking Status']=='Cancelled by Customer').sum():,}",
        f"{(df['Booking Status']=='Incomplete').sum():,}",
        f"{df.isnull().sum().sum():,}",
        str(df['Booking ID'].duplicated().sum()),
        f"{df['Booking Value'].mean():.2f}",
        f"{df['Ride Distance'].mean():.2f}"
    ]
}
pd.DataFrame(summary).to_csv('/content/01_extraction_summary.csv', index=False)
print('saved.')

saved.
